**Prática: Documental x Relacional na prática (MongoDB/TinyDB x SQLite/PostgreSQL)**

**Objetivo**

Comparar modelagem, consultas e desempenho entre bancos orientados a documentos e banco relacional usando o mesmo conjunto de dados e as mesmas operações.

**Competências trabalhadas**

1. Modelagem de dados em paradigmas diferentes.
2. Operações CRUD e consultas.
3. Medição de desempenho com método.
4. Análise crítica de trade-offs técnicos.

**Ferramentas**

1. Python 3.
2. TinyDB e/ou MongoDB.
3. SQLite (mais simples para laboratório local) ou PostgreSQL.
4. Matplotlib para visualização.
5. time/perf_counter para benchmark.

**Dataset**

Usar o mesmo conjunto de usuários com:

1. Dados básicos: ID, nome, cidade.
2. Estrutura aninhada: endereço.
3. Lista aninhada: pedidos com valor e data.
4. Volume recomendado: 10 mil a 100 mil registros (escalonado por grupo).


**Roteiro da prática**

1. Preparação

- Gerar dataset sintético em JSON.

In [2]:
import json
import random
from datetime import date, timedelta
from pathlib import Path

random.seed(42)

N_USERS = 10_000  # ajuste para 100_000 se quiser escalar
MAX_PEDIDOS_POR_USUARIO = 8

nomes = [
    "Ana", "Bruno", "Carla", "Daniel", "Eduarda", "Felipe", "Gabriela",
    "Henrique", "Isabela", "João", "Karina", "Lucas", "Mariana", "Nicolas",
    "Olivia", "Paulo", "Renata", "Samuel", "Tatiane", "Vinicius"
]
sobrenomes = ["Silva", "Souza", "Oliveira", "Santos", "Lima", "Costa", "Pereira", "Almeida"]
cidades = ["São Paulo", "Rio de Janeiro", "Belo Horizonte", "Curitiba", "Porto Alegre", "Recife", "Salvador"]
estados = ["SP", "RJ", "MG", "PR", "RS", "PE", "BA"]
bairros = ["Centro", "Jardins", "Copacabana", "Savassi", "Batel", "Moinhos", "Boa Viagem", "Pelourinho"]
ruas = ["Rua das Flores", "Av. Brasil", "Rua da Paz", "Av. Central", "Rua do Sol", "Rua das Palmeiras"]

def data_aleatoria(inicio=date(2022, 1, 1), fim=date(2025, 12, 31)):
    delta = (fim - inicio).days
    return (inicio + timedelta(days=random.randint(0, delta))).isoformat()

users = []
pedido_seq = 1

for uid in range(1, N_USERS + 1):
    cidade = random.choice(cidades)
    estado = random.choice(estados)
    n_pedidos = random.randint(0, MAX_PEDIDOS_POR_USUARIO)

    pedidos = []
    for _ in range(n_pedidos):
        pedidos.append({
            "pedido_id": pedido_seq,
            "valor": round(random.uniform(20, 2000), 2),
            "data": data_aleatoria()
        })
        pedido_seq += 1

    users.append({
        "id": uid,
        "nome": f"{random.choice(nomes)} {random.choice(sobrenomes)}",
        "cidade": cidade,
        "endereco": {
            "rua": random.choice(ruas),
            "numero": random.randint(1, 9999),
            "bairro": random.choice(bairros),
            "cidade": cidade,
            "estado": estado,
            "cep": f"{random.randint(10000, 99999)}-{random.randint(100, 999)}"
        },
        "pedidos": pedidos
    })

output_path = Path("usuarios_dataset.json")
output_path.write_text(json.dumps(users, ensure_ascii=False), encoding="utf-8")

print(f"Dataset gerado com {len(users)} usuários em: {output_path.resolve()}")
print(f"Total de pedidos gerados: {pedido_seq - 1}")

Dataset gerado com 10000 usuários em: F:\IbmecRepos\26.1\BDCC_ADS_26.1_8001\docs\bigdata\relacional\usuarios_dataset.json
Total de pedidos gerados: 39809


- Criar versão tabular para relacional:
  - tabela usuarios
  - tabela pedidos (FK para usuarios)

In [3]:
import sqlite3

db_path = "usuarios_relacional.db"

conn = sqlite3.connect(db_path)
conn.execute("PRAGMA foreign_keys = ON;")
cur = conn.cursor()

cur.executescript("""
DROP TABLE IF EXISTS pedidos;
DROP TABLE IF EXISTS usuarios;

CREATE TABLE usuarios (
    id INTEGER PRIMARY KEY,
    nome TEXT NOT NULL,
    cidade TEXT NOT NULL,
    rua TEXT,
    numero INTEGER,
    bairro TEXT,
    estado TEXT,
    cep TEXT
);

CREATE TABLE pedidos (
    pedido_id INTEGER PRIMARY KEY,
    usuario_id INTEGER NOT NULL,
    valor REAL NOT NULL,
    data TEXT NOT NULL,
    FOREIGN KEY (usuario_id) REFERENCES usuarios(id) ON DELETE CASCADE
);
""")

usuarios_rows = []
pedidos_rows = []

t0 = perf_counter()

for u in users:
    e = u["endereco"]
    usuarios_rows.append(
        (u["id"], u["nome"], u["cidade"], e["rua"], e["numero"], e["bairro"], e["estado"], e["cep"])
    )
    for p in u["pedidos"]:
        pedidos_rows.append((p["pedido_id"], u["id"], p["valor"], p["data"]))

cur.executemany("""
    INSERT INTO usuarios (id, nome, cidade, rua, numero, bairro, estado, cep)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""", usuarios_rows)

cur.executemany("""
    INSERT INTO pedidos (pedido_id, usuario_id, valor, data)
    VALUES (?, ?, ?, ?)
""", pedidos_rows)

conn.commit()

print(f"Banco criado: {db_path}")
print(f"Usuarios inseridos: {len(usuarios_rows)}")
print(f"Pedidos inseridos: {len(pedidos_rows)}")

NameError: name 'perf_counter' is not defined

2. Carga de dados

- Inserir no TinyDB ou MongoDB.

In [8]:
from tinydb import TinyDB
from time import perf_counter

tinydb_path = "usuarios_tinydb.json"

# remove arquivo anterior para evitar duplicidade em reexecuções
tinydb_file = Path(tinydb_path)
if tinydb_file.exists():
    tinydb_file.unlink()

db_tiny = TinyDB(tinydb_path)
usuarios_tiny = db_tiny.table("usuarios")

t0 = perf_counter()
doc_ids = usuarios_tiny.insert_multiple(users)
tempo_carga_tiny_ms = (perf_counter() - t0) * 1000

print(f"TinyDB criado em: {tinydb_file.resolve()}")
print(f"Usuários inseridos: {len(doc_ids)}")
print(f"Tempo total de carga: {tempo_carga_tiny_ms:.2f} ms")

PermissionError: [WinError 32] O arquivo já está sendo usado por outro processo: 'usuarios_tinydb.json'

- Registrar tempo total de carga.


In [6]:
from time import perf_counter

# Re-registra o tempo de carga no SQLite (usando os dados já gerados)
t0 = perf_counter()

cur.execute("DELETE FROM pedidos;")
cur.execute("DELETE FROM usuarios;")

cur.executemany("""
    INSERT INTO usuarios (id, nome, cidade, rua, numero, bairro, estado, cep)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""", usuarios_rows)

cur.executemany("""
    INSERT INTO pedidos (pedido_id, usuario_id, valor, data)
    VALUES (?, ?, ?, ?)
""", pedidos_rows)

conn.commit()
tempo_carga_sqlite_ms = (perf_counter() - t0) * 1000

# Registro consolidado dos tempos de carga
tempos_carga_ms = {
    "TinyDB": tempo_carga_tiny_ms,
    "SQLite": tempo_carga_sqlite_ms
}

print("Tempos totais de carga (ms):")
for banco, tempo in tempos_carga_ms.items():
    print(f"- {banco}: {tempo:.2f} ms")

Tempos totais de carga (ms):
- TinyDB: 96.32 ms
- SQLite: 0.30 ms



3. Consultas obrigatórias

- Q1: Buscar usuário por ID.
- Q2: Listar usuários por cidade.
- Q3: Total de pedidos por usuário.
- Q4: Top 10 usuários por valor total de pedidos.
- Q5: Atualizar endereço de um usuário.
- Q6: Remover usuário e seus pedidos.

4. Benchmark

- Executar cada operação 30 vezes.
- Descartar primeira execução (warm-up).
- Medir média, mediana e desvio padrão.
- Comparar sem índice e com índice (quando aplicável).

5. Visualização

- Gráficos de barras por operação.
- Gráfico de tempo de carga.
- Discussão sobre variação dos resultados.

**Template de tabela de resultados**

| Operação         | Banco      | Índice | Média (ms) | Mediana (ms) | Desvio padrão (ms) |
| ---------------- | ---------- | ------ | ---------: | -----------: | -----------------: |
| Q1 Buscar por ID | Documental | Não    |            |              |                    |
| Q1 Buscar por ID | Relacional | Não    |            |              |                    |
| Q1 Buscar por ID | Documental | Sim    |            |              |                    |
| Q1 Buscar por ID | Relacional | Sim    |            |              |                    |
| ...              | ...        | ...    |        ... |          ... |                ... |

**Perguntas para discussão**

1. Em quais operações o modelo relacional foi superior? Por quê?
2. Em quais operações o modelo documental ficou mais simples ou rápido?
3. Como a presença de dados aninhados impactou cada modelagem?
4. Qual solução você escolheria para:

- e-commerce
- app de conteúdo
- sistema financeiro

5. O que mudou ao adicionar índices?

**Entregáveis**

1. Código da carga e consultas.
2. Tabela de benchmark preenchida.
3. Gráficos.
4. Relatório curto (1 a 2 páginas) com conclusão técnica.

**Critérios de avaliação (sugestão)**

1. Correção técnica das consultas (30%).
2. Qualidade da medição experimental (30%).
3. Análise crítica dos resultados (30%).
4. Organização e clareza da entrega (10%).

Se quiser, eu já te entrego uma versão “copiar e colar” em markdown para entrar direto no plano de aula com seção de objetivos, recursos, procedimento e rubrica.
